    # Stateful GRU gait recognition - Combined Right Pocket

This notebook predicts an enrolled subject from every incoming accelerometer row using a **causal, two-layer stateful GRU**. It evaluates Session 1 -> Session 2 and Session 2 -> Session 1, with chronological validation tails and a GPU row-wise MLP baseline. The test session is never used for scaling, model selection, or early stopping.

The CSV is read in chunks once and cached as per-recording `.npy` arrays. A recording is one `(Subject, Session, Label)` run, additionally split at large timestamp gaps. Hidden state persists across all ordinary TBPTT chunks; it resets only at actual stream boundaries. No overlapping windows are constructed.

Run cells from top to bottom. Training requires CUDA-enabled PyTorch; the notebook intentionally stops if CUDA is unavailable.

## 1. Imports and CUDA-only configuration

The model, hidden state, minibatches, loss, optimizer, validation, and inference all run on CUDA. CPU use is limited to CSV/cache I/O and metric aggregation.

In [ ]:
import gc
import json
import random
import time
import warnings
from collections import defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from IPython.display import display

warnings.filterwarnings("default")
SEED = 41
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA is required for this notebook. Install/select a CUDA-enabled "
        "PyTorch environment and a working NVIDIA driver; CPU training is disabled."
    )

DEVICE = torch.device("cuda:0")
torch.cuda.set_device(DEVICE)
GPU_PROPERTIES = torch.cuda.get_device_properties(DEVICE)
FREE_VRAM_BYTES, TOTAL_VRAM_BYTES = torch.cuda.mem_get_info(DEVICE)
print(f"PyTorch: {torch.__version__}")
print(f"CUDA runtime: {torch.version.cuda}")
print(f"GPU: {GPU_PROPERTIES.name}")
print(f"Total VRAM: {TOTAL_VRAM_BYTES / 1024**3:.2f} GB")
print(f"Available VRAM: {FREE_VRAM_BYTES / 1024**3:.2f} GB")
print(f"Model device: {DEVICE}")

AMP_ENABLED = True
if FREE_VRAM_BYTES >= 20 * 1024**3:
    TBPTT_LENGTH = 2048
elif FREE_VRAM_BYTES >= 8 * 1024**3:
    TBPTT_LENGTH = 1024
else:
    TBPTT_LENGTH = 512
GRAD_ACCUM_STEPS = 1
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 4
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
GRAD_CLIP_NORM = 1.0
VALIDATION_FRACTION = 0.20
MAX_GAP_SECONDS = 0.10
MLP_EPOCHS = 10
MLP_ROWS_PER_SEGMENT = 3000
MLP_BATCH_SIZE = 2048

print(
    f"TBPTT rows={TBPTT_LENGTH}, AMP={AMP_ENABLED}, "
    f"gradient accumulation={GRAD_ACCUM_STEPS}"
)
print(
    "CUDA allocation before model creation:",
    f"{torch.cuda.memory_allocated(DEVICE) / 1024**3:.3f} GB",
)

## 2. Dataset configuration and schema

The source is `right_pocket_sessions_1_2_combined.csv`. The actual schema is validated rather than inferred or renamed. Sensor inputs are X, Y, Z, and acceleration magnitude.

In [ ]:
CANDIDATE_DATA_PATHS = [
    Path.cwd() / "right_pocket_sessions_1_2_combined.csv",
    Path.cwd() / "Combined_RightPocket" / "right_pocket_sessions_1_2_combined.csv",
]
DATA_PATH = next((path for path in CANDIDATE_DATA_PATHS if path.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not locate right_pocket_sessions_1_2_combined.csv. "
        "Run the notebook from Combined_RightPocket or its parent directory."
    )

DATA_DIR = DATA_PATH.parent
CACHE_ROOT = DATA_DIR / "stateful_gru_cache"
ARTIFACT_ROOT = DATA_DIR / "stateful_gru_artifacts"
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

REQUIRED_COLUMNS = ["Timestamp", "X", "Y", "Z", "Label", "Subject", "Session"]
CSV_COLUMNS = pd.read_csv(DATA_PATH, nrows=0).columns.tolist()
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(CSV_COLUMNS))
if missing_columns:
    raise ValueError(f"CSV is missing required columns: {missing_columns}")

SOURCE_STAT = DATA_PATH.stat()
CACHE_DIR = CACHE_ROOT / f"{DATA_PATH.stem}_{SOURCE_STAT.st_size}_{SOURCE_STAT.st_mtime_ns}"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = CACHE_DIR / "manifest.json"
print(f"Dataset: {DATA_PATH}")
print(f"CSV columns: {CSV_COLUMNS}")
print(f"CSV size: {SOURCE_STAT.st_size / 1024**2:.1f} MB")
print(f"Preprocessing cache: {CACHE_DIR}")

## 3. Chunked data-quality checks and chronological preprocessing

Rows with missing required values are counted and excluded; invalid identifiers or non-finite numeric values fail loudly. Exact duplicate sensor records are removed within each recording. Each recording is stably sorted by timestamp, and gaps greater than 0.10 seconds split the stream. The cache stores features and timestamps as separate memory-mapped arrays so epochs never reread the large CSV.

In [ ]:
def _cache_is_complete(manifest):
    if not manifest or manifest.get("source_size") != SOURCE_STAT.st_size:
        return False
    if manifest.get("source_mtime_ns") != SOURCE_STAT.st_mtime_ns:
        return False
    return all(
        (CACHE_DIR / entry["feature_file"]).is_file()
        and (CACHE_DIR / entry["timestamp_file"]).is_file()
        for entry in manifest.get("segments", [])
    )


def build_or_load_cache():
    if MANIFEST_PATH.is_file():
        with MANIFEST_PATH.open("r", encoding="utf-8") as file:
            cached_manifest = json.load(file)
        if _cache_is_complete(cached_manifest):
            return cached_manifest

    fragments = defaultdict(list)
    missing_counts = {column: 0 for column in REQUIRED_COLUMNS}
    raw_rows = 0
    discarded_missing_rows = 0
    parsed_rows = 0
    started = time.perf_counter()
    dtypes = {
        "Timestamp": "float64",
        "X": "float32",
        "Y": "float32",
        "Z": "float32",
        "Label": "string",
        "Subject": "float32",
        "Session": "float32",
    }

    for frame in pd.read_csv(
        DATA_PATH,
        usecols=REQUIRED_COLUMNS,
        dtype=dtypes,
        chunksize=500_000,
        low_memory=False,
    ):
        raw_rows += len(frame)
        for column in REQUIRED_COLUMNS:
            missing_counts[column] += int(frame[column].isna().sum())
        missing_mask = frame[REQUIRED_COLUMNS].isna().any(axis=1)
        discarded_missing_rows += int(missing_mask.sum())
        frame = frame.loc[~missing_mask, REQUIRED_COLUMNS]
        if frame.empty:
            continue

        numeric = frame[["Timestamp", "X", "Y", "Z", "Subject", "Session"]].to_numpy(
            dtype=np.float64, copy=False
        )
        if not np.isfinite(numeric).all():
            raise ValueError("Non-finite timestamp, feature, subject, or session found.")
        if not np.equal(frame["Subject"].to_numpy() % 1, 0).all():
            raise ValueError("Subject IDs must be integer-valued.")
        if not np.equal(frame["Session"].to_numpy() % 1, 0).all():
            raise ValueError("Session IDs must be integer-valued.")
        frame = frame.copy()
        frame["Subject"] = frame["Subject"].astype(np.int16)
        frame["Session"] = frame["Session"].astype(np.int8)
        if not frame["Session"].isin([1, 2]).all():
            raise ValueError("Expected Session values 1 and 2 only.")
        if frame["Label"].str.strip().eq("").any():
            raise ValueError("Empty activity labels found.")

        for key, group in frame.groupby(
            ["Subject", "Session", "Label"], sort=False, observed=True
        ):
            arr = group[["Timestamp", "X", "Y", "Z"]].to_numpy(
                dtype=np.float64, copy=True
            )
            fragments[(int(key[0]), int(key[1]), str(key[2]))].append(arr)
        parsed_rows += len(frame)

    segment_entries = []
    duplicate_rows = 0
    gap_splits = 0
    segment_number = 0
    for key in sorted(fragments):
        recording = np.concatenate(fragments.pop(key), axis=0)
        order = np.argsort(recording[:, 0], kind="stable")
        recording = recording[order]
        exact_duplicate = np.zeros(len(recording), dtype=bool)
        if len(recording) > 1:
            exact_duplicate[1:] = np.all(recording[1:] == recording[:-1], axis=1)
        duplicate_rows += int(exact_duplicate.sum())
        recording = recording[~exact_duplicate]
        if not len(recording):
            continue

        time_deltas = np.diff(recording[:, 0])
        split_points = np.flatnonzero(
            (time_deltas <= 0) | (time_deltas > MAX_GAP_SECONDS)
        ) + 1
        gap_splits += int(np.count_nonzero(time_deltas > MAX_GAP_SECONDS))
        boundaries = np.concatenate(([0], split_points, [len(recording)]))
        for start, stop in zip(boundaries[:-1], boundaries[1:]):
            part = recording[start:stop]
            if not len(part):
                continue
            xyz = part[:, 1:4].astype(np.float32, copy=False)
            magnitude = np.sqrt(np.sum(xyz * xyz, axis=1)).astype(np.float32)
            features = np.column_stack((xyz, magnitude)).astype(np.float32, copy=False)
            feature_name = f"features_{segment_number:05d}.npy"
            timestamp_name = f"timestamps_{segment_number:05d}.npy"
            np.save(CACHE_DIR / feature_name, features, allow_pickle=False)
            np.save(CACHE_DIR / timestamp_name, part[:, 0].astype(np.float64), allow_pickle=False)
            segment_entries.append(
                {
                    "subject": key[0],
                    "session": key[1],
                    "activity": key[2],
                    "segment_id": segment_number,
                    "n_rows": int(len(part)),
                    "feature_file": feature_name,
                    "timestamp_file": timestamp_name,
                }
            )
            segment_number += 1

    manifest = {
        "source_size": SOURCE_STAT.st_size,
        "source_mtime_ns": SOURCE_STAT.st_mtime_ns,
        "columns": CSV_COLUMNS,
        "raw_rows": raw_rows,
        "usable_rows": parsed_rows - duplicate_rows,
        "discarded_missing_rows": discarded_missing_rows,
        "missing_counts": missing_counts,
        "duplicate_rows_removed": duplicate_rows,
        "timestamp_gap_splits": gap_splits,
        "max_gap_seconds": MAX_GAP_SECONDS,
        "segments": segment_entries,
        "preprocessing_seconds": time.perf_counter() - started,
    }
    with MANIFEST_PATH.open("w", encoding="utf-8") as file:
        json.dump(manifest, file, indent=2)
    return manifest


MANIFEST = build_or_load_cache()
SEGMENTS = []
for entry in MANIFEST["segments"]:
    SEGMENTS.append(
        {
            **entry,
            "features": np.load(CACHE_DIR / entry["feature_file"], mmap_mode="r"),
            "timestamps": np.load(CACHE_DIR / entry["timestamp_file"], mmap_mode="r"),
        }
    )
print(f"Raw rows: {MANIFEST['raw_rows']:,}")
print(f"Usable rows: {MANIFEST['usable_rows']:,}")
print(f"Missing rows discarded: {MANIFEST['discarded_missing_rows']:,}")
print(f"Exact duplicate rows removed: {MANIFEST['duplicate_rows_removed']:,}")
print(f"Timestamp-gap splits: {MANIFEST['timestamp_gap_splits']:,}")
print(f"Cached independent sequences: {len(SEGMENTS):,}")
print(f"Preprocessing/cache load time: {MANIFEST['preprocessing_seconds']:.1f} s")

## 4. Dataset inspection and shared-subject population

The `Label` column denotes separate G/R activity recordings, not the identity target. The target is `Subject`. Subjects found in both sessions form the primary cross-session evaluation population; one-session subjects can be extra training classes only when they belong to the training session.

In [ ]:
SESSION_SUBJECTS = {
    session: {segment["subject"] for segment in SEGMENTS if segment["session"] == session}
    for session in (1, 2)
}
COMMON_SUBJECTS = sorted(SESSION_SUBJECTS[1] & SESSION_SUBJECTS[2])
if not COMMON_SUBJECTS:
    raise ValueError("No subjects have recordings in both sessions.")

print("Segments by session:")
display(
    pd.DataFrame(SEGMENTS)
    .groupby(["session", "activity"], observed=True)["n_rows"]
    .agg(["count", "sum"])
)
print(f"Session 1 subjects: {len(SESSION_SUBJECTS[1])}")
print(f"Session 2 subjects: {len(SESSION_SUBJECTS[2])}")
print(f"Primary shared subjects ({len(COMMON_SUBJECTS)}): {COMMON_SUBJECTS}")
print("Data quality summary:")
display(
    pd.Series(
        {
            "source rows": MANIFEST["raw_rows"],
            "usable rows": MANIFEST["usable_rows"],
            "missing rows removed": MANIFEST["discarded_missing_rows"],
            "exact duplicates removed": MANIFEST["duplicate_rows_removed"],
            "timestamp discontinuities split": MANIFEST["timestamp_gap_splits"],
        },
        name="count",
    ).to_frame()
)

## 5. Cross-session split and training-only normalization

For each direction, all subjects recorded in the training session are model classes. Each training-session sequence with at least two rows is split chronologically: its first 80% is training and last 20% is validation. A one-row gap fragment stays in training so its row is not discarded. Mean and standard deviation use only training rows. Test rows come only from the opposite session and only from shared subjects. Extra training-only classes are excluded from test predictions by restricting test logits to shared-subject classes.

In [ ]:
def build_direction_data(train_session, test_session):
    train_subjects = sorted(SESSION_SUBJECTS[train_session])
    subject_to_class = {subject: index for index, subject in enumerate(train_subjects)}
    common_class_positions = np.asarray(
        [subject_to_class[subject] for subject in COMMON_SUBJECTS], dtype=np.int64
    )
    train_items, validation_items, test_items = [], [], []

    for segment in SEGMENTS:
        subject = segment["subject"]
        session = segment["session"]
        n_rows = segment["n_rows"]
        if session == train_session:
            item = {**segment, "target": subject_to_class[subject]}
            if n_rows == 1:
                # Keep singleton gap-split fragments in training so their row loss is not dropped.
                train_items.append({**item, "start": 0, "stop": 1})
                continue
            cut = int(n_rows * (1.0 - VALIDATION_FRACTION))
            if cut < 1 or cut >= n_rows:
                raise ValueError(f"Invalid chronological split for sequence: {segment}")
            train_items.append({**item, "start": 0, "stop": cut})
            validation_items.append({**item, "start": cut, "stop": n_rows})
        elif session == test_session and subject in COMMON_SUBJECTS:
            test_items.append(
                {
                    **segment,
                    "start": 0,
                    "stop": n_rows,
                    "target": COMMON_SUBJECTS.index(subject),
                }
            )

    if not train_items or not validation_items or not test_items:
        raise ValueError("Training, validation, and cross-session test splits must be non-empty.")

    feature_sum = np.zeros(4, dtype=np.float64)
    feature_sum_sq = np.zeros(4, dtype=np.float64)
    training_rows = 0
    for item in train_items:
        block = np.asarray(item["features"][item["start"]:item["stop"]], dtype=np.float64)
        feature_sum += block.sum(axis=0)
        feature_sum_sq += np.square(block).sum(axis=0)
        training_rows += len(block)
    mean = feature_sum / training_rows
    variance = np.maximum(feature_sum_sq / training_rows - np.square(mean), 1e-12)
    std = np.sqrt(variance)
    return {
        "train_session": train_session,
        "test_session": test_session,
        "train_subjects": train_subjects,
        "subject_to_class": subject_to_class,
        "common_class_positions": common_class_positions,
        "train_items": train_items,
        "validation_items": validation_items,
        "test_items": test_items,
        "mean": mean.astype(np.float32),
        "std": std.astype(np.float32),
        "training_rows": training_rows,
    }


DIRECTION_DATA = {
    (1, 2): build_direction_data(1, 2),
    (2, 1): build_direction_data(2, 1),
}
for direction, split in DIRECTION_DATA.items():
    print(
        f"Session {direction[0]} -> {direction[1]} | "
        f"classes={len(split['train_subjects'])}, "
        f"train rows={split['training_rows']:,}, "
        f"validation rows={sum(i['stop'] - i['start'] for i in split['validation_items']):,}, "
        f"test rows={sum(i['stop'] - i['start'] for i in split['test_items']):,}"
    )
    print("Training class map (class index -> Subject):", split["train_subjects"])
    print("Training-only feature mean:", split["mean"])
    print("Training-only feature std:", split["std"])

## 6. Stateful causal GRU architecture

The two unidirectional GRUs use separate hidden states. Every chunk's hidden state is detached only after that chunk's backward pass, then passed unchanged into the next chronological chunk. Hidden states are initialized to `None` for each independent subject/session/activity/gap-split recording. Cross entropy is computed for every row in every training chunk.

In [ ]:
class StatefulGRU(nn.Module):
    def __init__(self, input_size, num_classes, hidden1=128, hidden2=64):
        super().__init__()
        self.gru1 = nn.GRU(input_size, hidden1, batch_first=True, bidirectional=False)
        self.dropout_between = nn.Dropout(0.2)
        self.gru2 = nn.GRU(hidden1, hidden2, batch_first=True, bidirectional=False)
        self.hidden_projection = nn.Linear(hidden2, 64)
        self.dropout_head = nn.Dropout(0.3)
        self.classifier = nn.Linear(64, num_classes)

    def forward(self, inputs, hidden=None):
        if hidden is None:
            hidden1 = hidden2 = None
        else:
            hidden1, hidden2 = hidden
        output, hidden1 = self.gru1(inputs, hidden1)
        output = self.dropout_between(output)
        output, hidden2 = self.gru2(output, hidden2)
        output = F.relu(self.hidden_projection(output))
        output = self.dropout_head(output)
        logits = self.classifier(output)
        return logits, (hidden1, hidden2)


def detach_hidden(hidden):
    return tuple(state.detach() for state in hidden)


def make_amp_scaler():
    return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)


_gpu_probe = StatefulGRU(4, max(2, len(COMMON_SUBJECTS))).to(DEVICE)
_probe_x = torch.zeros((1, 8, 4), device=DEVICE)
with torch.no_grad(), torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
    _probe_logits, _probe_hidden = _gpu_probe(_probe_x)
assert _probe_logits.device.type == "cuda"
assert all(state.device.type == "cuda" for state in _probe_hidden)
print("Model parameter device:", next(_gpu_probe.parameters()).device)
print("Training tensors device:", _probe_x.device, _probe_logits.device)
print(f"CUDA memory allocated: {torch.cuda.memory_allocated(DEVICE) / 1024**3:.3f} GB")
del _gpu_probe, _probe_x, _probe_logits, _probe_hidden
gc.collect()
torch.cuda.empty_cache()

## 7. Stateful TBPTT training, validation, checkpointing, and resume

Sequences may be shuffled as whole independent recordings between epochs, never as individual rows. At an ordinary TBPTT boundary, hidden state is detached and carried forward; it is not reset. Validation starts at the held-out tail with a fresh state so its scores do not depend on training-mode dropout or stale training hidden state. Checkpoints are written after every epoch and contain optimizer/scheduler/scaler state for recovery.

In [ ]:
def _gpu_chunk(item, start, stop, mean, std):
    raw = np.array(item["features"][start:stop], dtype=np.float32, copy=True)
    inputs = torch.from_numpy(raw).unsqueeze(0).to(DEVICE, non_blocking=True)
    mean_tensor = torch.as_tensor(mean, dtype=torch.float32, device=DEVICE)
    std_tensor = torch.as_tensor(std, dtype=torch.float32, device=DEVICE)
    inputs = (inputs - mean_tensor) / std_tensor
    targets = torch.full((stop - start,), int(item["target"]), dtype=torch.long, device=DEVICE)
    assert inputs.device.type == "cuda" and targets.device.type == "cuda"
    return inputs, targets


def _evaluate_gru(model, items, mean, std, num_classes):
    model.eval()
    loss_sum = 0.0
    correct = 0
    count = 0
    with torch.inference_mode():
        for item in items:
            hidden = None
            # Validation is the chronological continuation of its training
            # recording, so replay the training prefix without gradients.
            for prefix_start in range(0, item["start"], TBPTT_LENGTH):
                prefix_stop = min(prefix_start + TBPTT_LENGTH, item["start"])
                prefix_inputs, _ = _gpu_chunk(
                    item, prefix_start, prefix_stop, mean, std
                )
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                    _, hidden = model(prefix_inputs, hidden)
                hidden = detach_hidden(hidden)
            for start in range(item["start"], item["stop"], TBPTT_LENGTH):
                stop = min(start + TBPTT_LENGTH, item["stop"])
                inputs, targets = _gpu_chunk(item, start, stop, mean, std)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                    logits, hidden = model(inputs, hidden)
                    loss = F.cross_entropy(
                        logits.reshape(-1, num_classes), targets, reduction="sum"
                    )
                predictions = logits.argmax(dim=-1).reshape(-1)
                loss_sum += float(loss.item())
                correct += int((predictions == targets).sum().item())
                count += len(targets)
                hidden = detach_hidden(hidden)
    return loss_sum / count, correct / count


def _save_training_checkpoint(path, model, optimizer, scheduler, scaler, epoch, history,
                              best_accuracy, best_epoch, epochs_without_improvement, split):
    checkpoint = {
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "epoch": epoch,
        "history": history,
        "best_validation_accuracy": best_accuracy,
        "best_epoch": best_epoch,
        "epochs_without_improvement": epochs_without_improvement,
        "train_session": split["train_session"],
        "test_session": split["test_session"],
        "subject_to_class": split["subject_to_class"],
        "mean": split["mean"],
        "std": split["std"],
        "config": {
            "tbptt_length": TBPTT_LENGTH,
            "learning_rate": LEARNING_RATE,
            "amp": AMP_ENABLED,
            "hidden_sizes": [128, 64],
        },
    }
    torch.save(checkpoint, path)


def fit_stateful_gru(split, resume_path=None):
    train_session, test_session = split["train_session"], split["test_session"]
    run_dir = ARTIFACT_ROOT / f"gru_session_{train_session}_to_{test_session}"
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path = run_dir / "last_training_checkpoint.pt"
    best_path = run_dir / "best_model.pt"
    model = StatefulGRU(4, len(split["train_subjects"])).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    scaler = make_amp_scaler()
    history = {"train_loss": [], "validation_loss": [], "validation_accuracy": []}
    best_accuracy = -1.0
    best_epoch = 0
    stale_epochs = 0
    start_epoch = 0

    if resume_path is not None:
        checkpoint_path = Path(resume_path)
        if not checkpoint_path.is_file():
            raise FileNotFoundError(f"Resume checkpoint does not exist: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
        if checkpoint["train_session"] != train_session:
            raise ValueError("Resume checkpoint belongs to a different training session.")
        if checkpoint["subject_to_class"] != split["subject_to_class"]:
            raise ValueError("Resume checkpoint has a different training subject mapping.")
        if not np.array_equal(np.asarray(checkpoint["mean"]), split["mean"]):
            raise ValueError("Resume checkpoint uses different training feature means.")
        if not np.array_equal(np.asarray(checkpoint["std"]), split["std"]):
            raise ValueError("Resume checkpoint uses different training feature scales.")
        model.load_state_dict(checkpoint["model_state_dict"])
        optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        scheduler.load_state_dict(checkpoint["scheduler_state_dict"])
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
        start_epoch = int(checkpoint["epoch"]) + 1
        history = checkpoint["history"]
        best_accuracy = float(checkpoint["best_validation_accuracy"])
        stale_epochs = int(checkpoint["epochs_without_improvement"])
        best_epoch = int(checkpoint["best_epoch"])

    rng = np.random.default_rng(SEED + train_session)
    for epoch in range(start_epoch, EPOCHS):
        epoch_started = time.perf_counter()
        model.train()
        ordered_items = list(split["train_items"])
        rng.shuffle(ordered_items)
        loss_sum = 0.0
        rows_seen = 0
        optimizer_steps = 0

        for item in ordered_items:
            hidden = None
            chunk_starts = list(range(item["start"], item["stop"], TBPTT_LENGTH))
            optimizer.zero_grad(set_to_none=True)
            for chunk_index, start in enumerate(chunk_starts):
                stop = min(start + TBPTT_LENGTH, item["stop"])
                group_start = (chunk_index // GRAD_ACCUM_STEPS) * GRAD_ACCUM_STEPS
                group_size = min(GRAD_ACCUM_STEPS, len(chunk_starts) - group_start)
                inputs, targets = _gpu_chunk(item, start, stop, split["mean"], split["std"])
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                    logits, hidden = model(inputs, hidden)
                    loss = F.cross_entropy(logits.reshape(-1, len(split["train_subjects"])), targets)
                scaler.scale(loss / group_size).backward()
                loss_sum += float(loss.detach().item()) * len(targets)
                rows_seen += len(targets)

                is_group_end = (
                    (chunk_index + 1) % GRAD_ACCUM_STEPS == 0
                    or chunk_index + 1 == len(chunk_starts)
                )
                if is_group_end:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)
                    optimizer_steps += 1
                hidden = detach_hidden(hidden)

        train_loss = loss_sum / rows_seen
        validation_loss, validation_accuracy = _evaluate_gru(
            model,
            split["validation_items"],
            split["mean"],
            split["std"],
            len(split["train_subjects"]),
        )
        scheduler.step(validation_accuracy)
        history["train_loss"].append(train_loss)
        history["validation_loss"].append(validation_loss)
        history["validation_accuracy"].append(validation_accuracy)
        improved = validation_accuracy > best_accuracy
        if improved:
            best_accuracy = validation_accuracy
            best_epoch = epoch + 1
            stale_epochs = 0
            torch.save(
                {
                    "model_state_dict": model.state_dict(),
                    "epoch": best_epoch,
                    "validation_accuracy": best_accuracy,
                },
                best_path,
            )
        else:
            stale_epochs += 1

        _save_training_checkpoint(
            last_path, model, optimizer, scheduler, scaler, epoch, history,
            best_accuracy, best_epoch, stale_epochs, split
        )
        print(
            f"Session {train_session} -> {test_session} | epoch {epoch + 1:02d}/{EPOCHS} "
            f"| train loss={train_loss:.4f} | val loss={validation_loss:.4f} "
            f"| val row accuracy={validation_accuracy:.4f} "
            f"| optimizer steps={optimizer_steps:,} "
            f"| elapsed={time.perf_counter() - epoch_started:.1f}s"
        )
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            print("Early stopping: validation accuracy did not improve.")
            break

    if not best_path.is_file():
        raise RuntimeError("Training finished without producing a best-validation checkpoint.")
    best_checkpoint = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(best_checkpoint["model_state_dict"])
    model.eval()
    return model, history, best_epoch, best_path, last_path

## 8. Run both Stateful GRU directions and evaluate held-out sessions

The row predictions are emitted chunk-by-chunk and copied back to CPU immediately; test probabilities are never accumulated on GPU. Test logits are restricted to the 36 shared subjects before argmax so extra training-only subjects cannot be predicted in the primary evaluation.

In [ ]:
def predict_gru_items(model, split, items):
    model.eval()
    common_positions = torch.as_tensor(
        split["common_class_positions"], dtype=torch.long, device=DEVICE
    )
    y_true_parts, y_pred_parts, per_segment = [], [], []
    rows = 0
    torch.cuda.synchronize(DEVICE)
    started = time.perf_counter()
    with torch.inference_mode():
        for item in items:
            hidden = None
            segment_truth = int(item["target"])
            segment_predictions = []
            for start in range(item["start"], item["stop"], TBPTT_LENGTH):
                stop = min(start + TBPTT_LENGTH, item["stop"])
                inputs, _ = _gpu_chunk(item, start, stop, split["mean"], split["std"])
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                    logits, hidden = model(inputs, hidden)
                    restricted_logits = logits.index_select(2, common_positions)
                predictions = restricted_logits.argmax(dim=-1).reshape(-1).cpu().numpy()
                segment_predictions.append(predictions)
                rows += len(predictions)
                hidden = detach_hidden(hidden)
            segment_predictions = np.concatenate(segment_predictions)
            segment_truths = np.full(len(segment_predictions), segment_truth, dtype=np.int64)
            y_true_parts.append(segment_truths)
            y_pred_parts.append(segment_predictions.astype(np.int64, copy=False))
            per_segment.append(
                {
                    "subject": int(COMMON_SUBJECTS[segment_truth]),
                    "truth": segment_truth,
                    "predictions": segment_predictions,
                }
            )
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    return {
        "y_true": np.concatenate(y_true_parts),
        "y_pred": np.concatenate(y_pred_parts),
        "per_segment": per_segment,
        "inference_seconds": elapsed,
        "inference_seconds_per_row": elapsed / rows,
        "rows": rows,
    }


GRU_RESULTS = {}
for direction, split in DIRECTION_DATA.items():
    torch.cuda.reset_peak_memory_stats(DEVICE)
    training_started = time.perf_counter()
    resume_checkpoint = None
    model, history, best_epoch, best_path, last_path = fit_stateful_gru(
        split, resume_path=resume_checkpoint
    )
    training_seconds = time.perf_counter() - training_started
    test_output = predict_gru_items(model, split, split["test_items"])
    test_output.update(
        {
            "model": model,
            "history": history,
            "best_epoch": best_epoch,
            "best_checkpoint": best_path,
            "last_checkpoint": last_path,
            "training_seconds": training_seconds,
            "peak_gpu_allocated_gb": torch.cuda.max_memory_allocated(DEVICE) / 1024**3,
            "split": split,
            "model_name": "Stateful GRU",
        }
    )
    GRU_RESULTS[direction] = test_output
    print(
        f"Session {direction[0]} -> {direction[1]} test: "
        f"row accuracy={accuracy_score(test_output['y_true'], test_output['y_pred']):.4f}, "
        f"inference={test_output['inference_seconds_per_row'] * 1e6:.1f} us/row, "
        f"peak allocated GPU memory={test_output['peak_gpu_allocated_gb']:.2f} GB"
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 9. Row-wise metrics and visualizations

Metrics are calculated at the individual-row level. The optional block-vote metric aggregates 1,000 consecutive predictions only within one independent test recording and is reported separately from row accuracy.

In [ ]:
def block_vote_accuracy(per_segment, block_size=1000):
    correct, total = 0, 0
    for segment in per_segment:
        predictions = segment["predictions"]
        for start in range(0, len(predictions), block_size):
            block = predictions[start:start + block_size]
            values, counts = np.unique(block, return_counts=True)
            voted = int(values[np.argmax(counts)])
            correct += int(voted == segment["truth"])
            total += 1
    return correct / total if total else float("nan")


metric_rows = []
for direction, result in GRU_RESULTS.items():
    y_true, y_pred = result["y_true"], result["y_pred"]
    metric_rows.append(
        {
            "direction": f"{direction[0]} -> {direction[1]}",
            "model": "Stateful GRU",
            "row_accuracy": accuracy_score(y_true, y_pred),
            "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
            "block_vote_accuracy_1000_rows": block_vote_accuracy(result["per_segment"]),
            "training_minutes": result["training_seconds"] / 60,
            "inference_microseconds_per_row": result["inference_seconds_per_row"] * 1e6,
            "peak_gpu_allocated_gb": result["peak_gpu_allocated_gb"],
        }
    )
    print(f"\n=== Stateful GRU: Session {direction[0]} -> Session {direction[1]} ===")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=np.arange(len(COMMON_SUBJECTS)),
            target_names=[str(subject) for subject in COMMON_SUBJECTS],
            zero_division=0,
            digits=4,
        )
    )
    per_subject = pd.DataFrame(
        {
            "Subject": COMMON_SUBJECTS,
            "row_accuracy": [
                float(np.mean(y_pred[y_true == class_index] == class_index))
                if np.any(y_true == class_index)
                else float("nan")
                for class_index in range(len(COMMON_SUBJECTS))
            ],
        }
    )
    display(per_subject)

METRICS = pd.DataFrame(metric_rows)
display(METRICS)

fig, axes = plt.subplots(1, 2, figsize=(18, 7), constrained_layout=True)
for ax, (direction, result) in zip(axes, GRU_RESULTS.items()):
    matrix = confusion_matrix(
        result["y_true"], result["y_pred"], labels=np.arange(len(COMMON_SUBJECTS))
    )
    image = ax.imshow(matrix, interpolation="nearest", cmap="Blues")
    ax.set_title(f"Row confusion matrix: {direction[0]} -> {direction[1]}")
    ax.set_xlabel("Predicted Subject")
    ax.set_ylabel("True Subject")
    ax.set_xticks(np.arange(len(COMMON_SUBJECTS)))
    ax.set_yticks(np.arange(len(COMMON_SUBJECTS)))
    ax.set_xticklabels(COMMON_SUBJECTS, rotation=90, fontsize=7)
    ax.set_yticklabels(COMMON_SUBJECTS, fontsize=7)
    fig.colorbar(image, ax=ax, fraction=0.046, pad=0.04)
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4), constrained_layout=True)
for ax, (direction, result) in zip(axes, GRU_RESULTS.items()):
    history = result["history"]
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    ax.plot(epochs, history["train_loss"], label="Train loss")
    ax.plot(epochs, history["validation_loss"], label="Validation loss")
    ax.set_title(f"Session {direction[0]} -> {direction[1]}")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Cross-entropy")
    ax.legend()
plt.show()

chance_accuracy = 1.0 / len(COMMON_SUBJECTS)
print(f"Uniform chance accuracy over {len(COMMON_SUBJECTS)} shared subjects: {chance_accuracy:.4f}")

## 10. GPU row-wise MLP baseline

The MLP is an independent-row reference, not a recurrent model. It uses exactly the same direction-specific training rows, training-only normalization, validation tails, shared-subject test population, and test-logit restriction. Training samples are drawn only from the training prefixes and shuffled as independent rows.

In [ ]:
class RowWiseMLP(nn.Module):
    def __init__(self, input_size, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, inputs):
        return self.network(inputs)


def make_mlp_training_sample(split, seed):
    rng = np.random.default_rng(seed)
    feature_parts, target_parts = [], []
    for item in split["train_items"]:
        length = item["stop"] - item["start"]
        sample_count = min(length, MLP_ROWS_PER_SEGMENT)
        offsets = rng.choice(length, size=sample_count, replace=False)
        raw = np.asarray(item["features"][item["start"] + offsets], dtype=np.float32)
        normalized = (raw - split["mean"]) / split["std"]
        feature_parts.append(normalized)
        target_parts.append(np.full(sample_count, item["target"], dtype=np.int64))
    features = np.concatenate(feature_parts).astype(np.float32, copy=False)
    targets = np.concatenate(target_parts)
    order = rng.permutation(len(targets))
    return features[order], targets[order]


def evaluate_mlp(model, items, split, restrict_to_common=False):
    model.eval()
    common_positions = torch.as_tensor(
        split["common_class_positions"], dtype=torch.long, device=DEVICE
    )
    predictions, truths = [], []
    started = time.perf_counter()
    torch.cuda.synchronize(DEVICE)
    with torch.inference_mode():
        for item in items:
            for start in range(item["start"], item["stop"], MLP_BATCH_SIZE):
                stop = min(start + MLP_BATCH_SIZE, item["stop"])
                raw = np.array(item["features"][start:stop], dtype=np.float32, copy=True)
                inputs = torch.from_numpy(raw).to(DEVICE, non_blocking=True)
                inputs = (
                    inputs - torch.as_tensor(split["mean"], device=DEVICE)
                ) / torch.as_tensor(split["std"], device=DEVICE)
                with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                    logits = model(inputs)
                    if restrict_to_common:
                        logits = logits.index_select(1, common_positions)
                predictions.append(logits.argmax(dim=1).cpu().numpy())
                truths.append(
                    np.full(stop - start, item["target"], dtype=np.int64)
                )
    torch.cuda.synchronize(DEVICE)
    elapsed = time.perf_counter() - started
    return np.concatenate(truths), np.concatenate(predictions), elapsed


def fit_mlp(split, seed):
    x_train, y_train = make_mlp_training_sample(split, seed)
    model = RowWiseMLP(4, len(split["train_subjects"])).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=2
    )
    scaler = make_amp_scaler()
    best_accuracy = -1.0
    best_state = None
    stale_epochs = 0
    history = {"train_loss": [], "validation_accuracy": []}
    rng = np.random.default_rng(seed)
    started = time.perf_counter()

    for epoch in range(MLP_EPOCHS):
        model.train()
        order = rng.permutation(len(y_train))
        loss_sum, rows_seen = 0.0, 0
        for start in range(0, len(order), MLP_BATCH_SIZE):
            indices = order[start:start + MLP_BATCH_SIZE]
            inputs = torch.from_numpy(x_train[indices]).to(DEVICE, non_blocking=True)
            targets = torch.from_numpy(y_train[indices]).to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.autocast("cuda", dtype=torch.float16, enabled=AMP_ENABLED):
                logits = model(inputs)
                loss = F.cross_entropy(logits, targets)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            loss_sum += float(loss.detach().item()) * len(targets)
            rows_seen += len(targets)

        _, validation_predictions, _ = evaluate_mlp(
            model, split["validation_items"], split, restrict_to_common=False
        )
        validation_truths = np.concatenate(
            [
                np.full(item["stop"] - item["start"], item["target"], dtype=np.int64)
                for item in split["validation_items"]
            ]
        )
        validation_accuracy = accuracy_score(validation_truths, validation_predictions)
        scheduler.step(validation_accuracy)
        history["train_loss"].append(loss_sum / rows_seen)
        history["validation_accuracy"].append(validation_accuracy)
        if validation_accuracy > best_accuracy:
            best_accuracy = validation_accuracy
            best_state = {
                name: tensor.detach().clone()
                for name, tensor in model.state_dict().items()
            }
            stale_epochs = 0
        else:
            stale_epochs += 1
        print(
            f"MLP epoch {epoch + 1:02d}/{MLP_EPOCHS} "
            f"| sampled rows={len(y_train):,} "
            f"| train loss={history['train_loss'][-1]:.4f} "
            f"| validation row accuracy={validation_accuracy:.4f}"
        )
        if stale_epochs >= EARLY_STOPPING_PATIENCE:
            break

    if best_state is None:
        raise RuntimeError("MLP training did not produce a validation checkpoint.")
    model.load_state_dict(best_state)
    model.eval()
    del x_train, y_train
    return model, history, time.perf_counter() - started


MLP_RESULTS = {}
for index, (direction, split) in enumerate(DIRECTION_DATA.items()):
    model, history, training_seconds = fit_mlp(split, SEED + 100 + index)
    y_true, y_pred, inference_seconds = evaluate_mlp(
        model, split["test_items"], split, restrict_to_common=True
    )
    MLP_RESULTS[direction] = {
        "model": model,
        "history": history,
        "y_true": y_true,
        "y_pred": y_pred,
        "training_seconds": training_seconds,
        "inference_seconds": inference_seconds,
        "inference_seconds_per_row": inference_seconds / len(y_true),
    }
    print(
        f"MLP Session {direction[0]} -> {direction[1]} "
        f"row accuracy={accuracy_score(y_true, y_pred):.4f}"
    )
    del model
    gc.collect()
    torch.cuda.empty_cache()

## 11. Comparable baseline table and cross-session summary

The MLP and GRU use identical test recordings and class restrictions. The previous windowed CNN is not retrained here; its historical scores are not inserted into this table because only results measured under this exact split and preprocessing would be comparable.

In [ ]:
comparison_rows = []
for direction in DIRECTION_DATA:
    for model_name, results in (
        ("Row-wise MLP", MLP_RESULTS[direction]),
        ("Stateful GRU", GRU_RESULTS[direction]),
    ):
        comparison_rows.append(
            {
                "direction": f"{direction[0]} -> {direction[1]}",
                "model": model_name,
                "row_accuracy": accuracy_score(results["y_true"], results["y_pred"]),
                "macro_precision": precision_score(
                    results["y_true"], results["y_pred"], average="macro", zero_division=0
                ),
                "macro_recall": recall_score(
                    results["y_true"], results["y_pred"], average="macro", zero_division=0
                ),
                "macro_f1": f1_score(
                    results["y_true"], results["y_pred"], average="macro", zero_division=0
                ),
                "training_minutes": results["training_seconds"] / 60,
                "inference_microseconds_per_row": (
                    results["inference_seconds_per_row"] * 1e6
                    if "inference_seconds_per_row" in results
                    else results["inference_seconds"] / results["rows"] * 1e6
                ),
            }
        )
COMPARISON = pd.DataFrame(comparison_rows)
display(COMPARISON)
print("Mean cross-session row accuracy by model:")
display(COMPARISON.groupby("model")["row_accuracy"].mean().rename("mean_row_accuracy"))
print(f"Chance level: {chance_accuracy:.4f} ({len(COMMON_SUBJECTS)} evaluated subjects)")

## 12. Save inference artifacts

Each direction gets its own best-validation model, class map, training-only normalization statistics, architecture/configuration, and run metadata. A separate epoch-boundary `last_training_checkpoint.pt` is retained to resume training by passing its path to `fit_stateful_gru` in Section 7.

In [ ]:
def save_inference_artifact(direction, result):
    split = result["split"]
    model = result["model"]
    run_dir = ARTIFACT_ROOT / f"gru_session_{direction[0]}_to_{direction[1]}"
    run_dir.mkdir(parents=True, exist_ok=True)
    artifact_path = run_dir / "inference_artifact.pt"
    artifact = {
        "model_state_dict": {
            name: tensor.detach().cpu()
            for name, tensor in model.state_dict().items()
        },
        "architecture": {
            "name": "StatefulGRU",
            "input_features": ["X", "Y", "Z", "acceleration_magnitude"],
            "hidden_sizes": [128, 64],
            "bidirectional": False,
            "dropout_between_grus": 0.2,
            "head_size": 64,
            "head_dropout": 0.3,
            "num_classes": len(split["train_subjects"]),
        },
        "subject_to_class": split["subject_to_class"],
        "class_to_subject": split["train_subjects"],
        "normalization_mean": split["mean"].tolist(),
        "normalization_std": split["std"].tolist(),
        "metadata": {
            "train_session": direction[0],
            "test_session": direction[1],
            "common_evaluation_subjects": COMMON_SUBJECTS,
            "best_epoch": result["best_epoch"],
            "validation_accuracy": result["history"]["validation_accuracy"][
                result["best_epoch"] - 1
            ],
            "training_seconds": result["training_seconds"],
            "source_csv": str(DATA_PATH),
        },
    }
    torch.save(artifact, artifact_path)
    json_path = run_dir / "inference_metadata.json"
    with json_path.open("w", encoding="utf-8") as file:
        json.dump(
            {key: value for key, value in artifact.items() if key != "model_state_dict"},
            file,
            indent=2,
        )
    print(f"Saved inference weights and metadata: {artifact_path}")
    return artifact_path


for direction, result in GRU_RESULTS.items():
    save_inference_artifact(direction, result)
print("Artifacts root:", ARTIFACT_ROOT)

## 13. Continuous row-wise inference demonstration

`predict_subject(sensor_row, hidden_state)` accepts one raw X/Y/Z reading (or X/Y/Z/magnitude), normalizes it with the saved training-session statistics, and returns the predicted raw Subject ID, class probabilities, and updated CUDA hidden state. Keep the hidden state for consecutive rows from the same recording; set it to `None` when a new subject/session/activity/recording begins or a timestamp discontinuity is detected.

In [ ]:
INFERENCE_DIRECTION = (1, 2)
INFERENCE_RESULT = GRU_RESULTS[INFERENCE_DIRECTION]
INFERENCE_SPLIT = INFERENCE_RESULT["split"]
INFERENCE_MODEL = INFERENCE_RESULT["model"]
INFERENCE_SUBJECTS = INFERENCE_SPLIT["train_subjects"]
INFERENCE_MEAN = torch.as_tensor(INFERENCE_SPLIT["mean"], device=DEVICE)
INFERENCE_STD = torch.as_tensor(INFERENCE_SPLIT["std"], device=DEVICE)


def predict_subject(sensor_row, hidden_state):
    values = np.asarray(sensor_row, dtype=np.float32).reshape(-1)
    if values.size == 3:
        magnitude = np.sqrt(np.sum(values * values)).astype(np.float32)
        values = np.concatenate((values, np.asarray([magnitude], dtype=np.float32)))
    elif values.size != 4:
        raise ValueError("sensor_row must contain X, Y, Z or X, Y, Z, acceleration magnitude.")
    if not np.isfinite(values).all():
        raise ValueError("sensor_row contains non-finite values.")

    inputs = torch.as_tensor(values, dtype=torch.float32, device=DEVICE).view(1, 1, 4)
    inputs = (inputs - INFERENCE_MEAN) / INFERENCE_STD
    if hidden_state is not None:
        hidden_state = tuple(state.to(DEVICE) for state in hidden_state)
    INFERENCE_MODEL.eval()
    with torch.inference_mode(), torch.autocast(
        "cuda", dtype=torch.float16, enabled=AMP_ENABLED
    ):
        logits, updated_hidden = INFERENCE_MODEL(inputs, hidden_state)
        probabilities = torch.softmax(logits[0, 0].float(), dim=-1)
        predicted_index = int(probabilities.argmax().item())
    probability_by_subject = {
        int(subject): float(probabilities[class_index].item())
        for class_index, subject in enumerate(INFERENCE_SUBJECTS)
    }
    return int(INFERENCE_SUBJECTS[predicted_index]), probability_by_subject, updated_hidden


example_segment = next(
    item for item in INFERENCE_SPLIT["test_items"]
    if item["session"] == INFERENCE_DIRECTION[1]
)
example_count = min(20, example_segment["n_rows"])
example_features = np.asarray(example_segment["features"][:example_count], dtype=np.float32)
example_timestamps = np.asarray(example_segment["timestamps"][:example_count])
hidden_state = None
stream_predictions = []
for row_index, features in enumerate(example_features):
    if row_index > 0 and (
        example_timestamps[row_index] - example_timestamps[row_index - 1]
        > MAX_GAP_SECONDS
    ):
        hidden_state = None
    subject_id, probabilities, hidden_state = predict_subject(features[:3], hidden_state)
    stream_predictions.append(subject_id)
print(
    f"Stream demo: Subject {example_segment['subject']}, "
    f"Session {example_segment['session']}, activity {example_segment['activity']}"
)
print("First 20 continuous row predictions:", stream_predictions)
print("Returned probability count:", len(probabilities))
print("Hidden-state devices:", [state.device for state in hidden_state])

## 14. Interpretation and reproducibility notes

- The reported accuracy is cross-session **row-wise subject accuracy**, not random-row accuracy.
- The optional 1,000-row block vote is an aggregated identity metric and is listed separately.
- Validation and all preprocessing statistics come only from the training session.
- Extra one-session subjects can be training classes but are excluded from the opposite-session test set and from test argmax.
- Activity labels are used to define separate streams, not as prediction features or target labels.
- No accuracy is assumed in advance; use the measured result tables and confusion matrices above.
- The GRU's hidden state must be retained for continuous inference and reset at real recording boundaries, never at ordinary TBPTT chunk boundaries.

The notebook uses four input features `[X, Y, Z, sqrt(X² + Y² + Z²)]`, a unidirectional 128-unit GRU followed by a unidirectional 64-unit GRU, then a 64-unit ReLU/dropout head and subject classifier. AMP, gradient clipping, AdamW, validation-based learning-rate reduction, early stopping, per-epoch resumable checkpoints, and CUDA-memory reporting are enabled.